# Branch Sync

First task of each wave in `git` and `hybrid` workflows (`w<wave>_sync`, then the If/else `w<wave>_synced`, then `w<wave>_sessions`). Turn it off with `"workflow": {"sync_branch": false}`.

The integrator commits and pushes from the `build_root` Git folder and never pulls. A fix pushed to the build branch from elsewhere (a person, or an external monitor agent repairing the run) is not in the folder until it is pulled, and the integrator's next push would be rejected as non-fast-forward. This task pulls the branch into the folder (Repos API, as the job's `run_as`) before the wave's sessions, so **repairing from `w<wave>_sync`** picks the fix up.

* The folder must already be on the build branch. The sync never switches branches.
* Uncommitted files (carryover from sessions that did not pass) are kept unless an incoming commit changes them. Then the pull fails and lists the conflicting files, and the folder is unchanged.
* Only push to the branch while no build run is active (the build job allows one run at a time; a repair can only start on a finished run).

Outputs: task value `synced` (`"true"` / `"false"`), read by `w<wave>_synced`, and `wave_<wave>_sync.json` in the run's status directory (commits and files pulled, or the error). The task never raises: a failure sets `synced` to `"false"`, the wave is Excluded, and `build_verdict` fails the run with the reason.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("wave", "", "Wave id")
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("branch", "", "Build branch")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

wave = dbutils.widgets.get("wave").strip()
run_id = dbutils.widgets.get("run_id").strip()
branch = dbutils.widgets.get("branch").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

print(f"wave={wave} run_id={run_id} branch={branch}")

In [0]:
import json
import os
import subprocess
import sys

# Helper modules sit next to this notebook (cwd for workspace notebooks).
sys.path.insert(0, os.getcwd())
from branch_sync import (branch_problem, git_folder_candidates, parse_git_log, pull_failure,
                         sync_message, sync_result, sync_result_path)


def _git(root, *args):
    """Read-only git CLI in the Git folder (CLI-enabled folders); None when unavailable."""
    try:
        p = subprocess.run(["git", "-C", root, *args], capture_output=True, text=True, timeout=60)
        return p.stdout if p.returncode == 0 else None
    except Exception:
        return None


# Never raise: a failed task would make w<wave>_synced Upstream failed, and the wave's
# run_if ALL_DONE tasks would still run on an unsynced folder. Every failure becomes synced=false.
result, out_path = None, None
try:
    if not (wave and run_id and branch and build_root and manifest_path):
        raise ValueError(f"wave, run_id, branch, build_root and manifest_path are required (got wave={wave!r}, "
                         f"run_id={run_id!r}, branch={branch!r}, build_root={build_root!r}, manifest_path={manifest_path!r})")
    with open(manifest_path) as f:
        manifest = json.load(f)
    status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
    os.makedirs(status_dir, exist_ok=True)
    out_path = sync_result_path(status_dir, wave)
    if os.path.exists(out_path):
        os.remove(out_path)  # an earlier attempt's result must not stand for this one

    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    repo, root = None, None
    for cand in git_folder_candidates(build_root):
        try:
            oid = w.workspace.get_status(cand).object_id
            repo, root = w.repos.get(oid), cand
            break
        except Exception:
            continue  # not a Git folder root; try the parent
    if repo is None:
        result = sync_result(wave, run_id, branch, ok=False,
                             error=f"no Git folder found at or above {build_root}; set workflow.sync_branch to false "
                                   f"for a build_root outside a Git folder")
    else:
        before = repo.head_commit_id
        problem = branch_problem(repo.branch, branch)
        if problem:
            result = sync_result(wave, run_id, branch, ok=False, before=before, error=problem, git_folder=root)
        else:
            try:
                w.repos.update(repo.id, branch=branch)
            except Exception as e:
                result = sync_result(wave, run_id, branch, ok=False, before=before, git_folder=root,
                                     **pull_failure(e))
            else:
                after = w.repos.get(repo.id).head_commit_id
                commits, files = [], []
                if before and after and before != after:
                    commits = parse_git_log(_git(root, "log", "--format=%H%x09%an%x09%s", f"{before}..{after}"))
                    files = [f for f in (_git(root, "diff", "--name-only", before, after) or "").splitlines() if f]
                result = sync_result(wave, run_id, branch, ok=True, before=before, after=after,
                                     commits=commits, files=files, git_folder=root)
except Exception as e:
    result = sync_result(wave, run_id, branch, ok=False, error=f"sync task error: {type(e).__name__}: {str(e)[:300]}")

message = sync_message(result)
print(json.dumps(result, indent=2, default=str))
print(message)

if out_path:
    try:
        with open(out_path, "w") as f:
            json.dump({**result, "message": message}, f, indent=2, default=str)
    except OSError as e:
        print(f"WARN: could not write {out_path}: {e}")
        if result["ok"]:
            # build_verdict reads this file; without it a later failure would be unexplained.
            result = {**result, "ok": False, "error": f"could not write {out_path}: {e}"}
            message = sync_message(result)
            print(message)

# No try/except: if this fails in a job, w<wave>_synced cannot resolve its value. Outside a job
# (interactive), taskValues.set does nothing.
dbutils.jobs.taskValues.set(key="synced", value="true" if result["ok"] else "false")
dbutils.notebook.exit(message)